In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from spark_session import get_spark

spark = get_spark("FastFood-P5-Streaming",
                  extra_packages=["org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"])
spark.sparkContext.setLogLevel("WARN")

https://repos.spark-packages.org added as a remote repository with the name: repo-1


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0c87c66e-7f6b-40c7-bd93-0603e42e4340;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.h

In [2]:
orders_schema = StructType([
    StructField("order_id", StringType()),
    StructField("restaurant_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("city", StringType()),
    StructField("status", StringType()),
    StructField("amount_mad", DoubleType()),
    StructField("items_count", IntegerType()),
    StructField("distance_km", DoubleType()),
    StructField("courier_id", StringType()),
    StructField("event_time", TimestampType()),
])

raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", "kafka:9092")
       .option("subscribe", "orders")
       .option("startingOffsets", "latest")
       .option("failOnDataLoss", "false")
       .load())
raw.printSchema()          # key, value (binary), topic, partition, offset, timestamp...

orders = (raw.select(F.col("value").cast("string").alias("json"),
                     "partition", "offset", F.col("timestamp").alias("kafka_ts"))
             .withColumn("o", F.from_json("json", orders_schema,
                                          {"timestampFormat": "yyyy-MM-dd'T'HH:mm:ssX"}))
             .select("o.*", "partition", "offset", "kafka_ts"))
print("isStreaming :", orders.isStreaming)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

isStreaming : True


In [3]:
q_console = (orders.writeStream
             .queryName("p5_console_brut")
             .format("console")
             .outputMode("append")
             .option("truncate", False)
             .option("numRows", 10)
             .trigger(processingTime="10 seconds")
             .start())

26/11/15 18:30:14 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-389a1bcd-dcbe-4546-a6af-fa21318b5b7e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/15 18:30:14 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:30:15 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0007148|R-081        |C-01629    |Marrakech |CREATED  |111.92    |2          |4.5        |L-MAR-13  |2026-11-15 18:30:16|2        |2807  |2026-11-15 18:30:16.194|
|ORD-0004257|R-030        |C-04405    |Marrakech |PICKED_UP|351.04    |6          |1.4        |L-MAR-10  |2026-11-15 18:30:16|1        |2699  |2026-11-15 18:30:16.245|
|ORD-0004425|R-102        |C-03170    |Casablanca|PICKED_UP|178

26/11/15 18:30:25 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 10257 milliseconds
                                                                                

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0003039|R-192        |C-01258    |Rabat     |PICKED_UP|203.72    |4          |3.1        |L-RAB-09  |2026-11-15 18:30:16|2        |2808  |2026-11-15 18:30:16.398|
|ORD-0001793|R-015        |C-00373    |Casablanca|PICKED_UP|201.7     |4          |3.7        |L-CAS-05  |2026-11-15 18:30:16|2        |2809  |2026-11-15 18:30:16.551|
|ORD-0007152|R-179        |C-05172    |Casablanca|CREATED  |70.

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0002415|R-113        |C-00006    |Marrakech |PICKED_UP|48.67     |1          |3.6        |L-MAR-11  |2026-11-15 18:32:09|2        |3593  |2026-11-15 18:32:10.12 |
|ORD-0002135|R-004        |C-04559    |Rabat     |PICKED_UP|346.47    |5          |2.8        |L-RAB-14  |2026-11-15 18:32:10|2        |3594  |2026-11-15 18:32:10.171|
|ORD-0005836|R-153        |C-00703    |Marrakech |PICKED_UP|20

In [4]:
q_bronze = (raw.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value",
                           "topic", "partition", "offset", "timestamp")
            .writeStream
            .queryName("p5_bronze")
            .format("json")
            .option("path", "s3a://lake/bronze/orders_raw/")
            .option("checkpointLocation", "s3a://lake/checkpoints/p5_bronze_orders_raw/")
            .outputMode("append")
            .trigger(processingTime="30 seconds")
            .start())

26/11/15 18:32:25 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/15 18:32:27 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:32:27 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 14
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0008287|R-072        |C-00541    |Casablanca|CREATED  |324.54    |6          |6.0        |L-CAS-09  |2026-11-15 18:32:20|2        |3646  |2026-11-15 18:32:20.057|
|ORD-0008288|R-103        |C-04742    |Tanger    |CREATED  |168.5     |3          |4.3        |L-TAN-07  |2026-11-15 18:32:20|2        |3647  |2026-11-15 18:32:20.108|
|ORD-0004932|R-144        |C-05714    |Marrakech |PICKED_UP|14

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0001846|R-129        |C-00050    |Rabat     |PICKED_UP|200.46    |3          |3.2        |L-RAB-14  |2026-11-15 18:32:30|2        |3714  |2026-11-15 18:32:30.055|
|ORD-0003980|R-037        |C-00872    |Tanger    |PICKED_UP|404.95    |6          |5.1        |L-TAN-13  |2026-11-15 18:32:30|2        |3715  |2026-11-15 18:32:30.363|
|ORD-0006010|R-091        |C-03746    |Casablanca|PICKED_UP|26

In [5]:
q_counts = (orders.groupBy("city", "status").count()
            .orderBy("city", "status")
            .writeStream
            .queryName("p5_counts")
            .format("console")
            .outputMode("complete")
            .option("truncate", False)
            .trigger(processingTime="15 seconds")
            .start())

26/11/15 18:32:51 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-8d59afd8-0125-4e12-b9f4-af341ebccd8a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/15 18:32:51 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:32:51 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-------+-----+
|city     |status |count|
+---------+-------+-----+
|Marrakech|CREATED|1    |
+---------+-------+-----+

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0002608|R-106        |C-04583    |Casablanca|PICKED_UP|223.76    |4          |2.0        |L-CAS-01  |2026-11-15 18:32:49|2        |3862  |2026-11-15 18:32:50.067|
|O

-------------------------------------------
Batch: 1
-------------------------------------------
+----------+---------+-----+
|city      |status   |count|
+----------+---------+-----+
|Casablanca|CANCELLED|3    |
|Casablanca|CREATED  |14   |
|Casablanca|PICKED_UP|50   |
|Marrakech |CANCELLED|2    |
|Marrakech |CREATED  |11   |
|Marrakech |PICKED_UP|12   |
|Rabat     |CANCELLED|2    |
|Rabat     |CREATED  |13   |
|Rabat     |PICKED_UP|21   |
|Tanger    |CREATED  |10   |
|Tanger    |DELIVERED|1    |
|Tanger    |PICKED_UP|20   |
+----------+---------+-----+



-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0004839|R-058        |C-04302    |Casablanca|PICKED_UP|466.19    |6          |1.5        |L-CAS-04  |2026-11-15 18:32:59|2        |3924  |2026-11-15 18:33:00.108|
|ORD-0003341|R-041        |C-05911    |Casablanca|PICKED_UP|263.74    |5          |5.3        |L-CAS-01  |2026-11-15 18:33:00|2        |3925  |2026-11-15 18:33:00.367|
|ORD-0002661|R-110        |C-01877    |Marrakech |PICKED_UP|22

In [6]:
windowed = (orders
            .withWatermark("event_time", "10 minutes")
            .groupBy(F.window("event_time", "5 minutes"), "city")
            .count())

q_wm = (windowed.writeStream
        .queryName("p5_watermark")
        .format("console")
        .outputMode("update")
        .option("truncate", False)
        .trigger(processingTime="15 seconds")
        .start())

26/11/15 18:33:22 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-449fa809-7a19-4d67-8eb6-d519180266a8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/11/15 18:33:22 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/11/15 18:33:22 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+------+-----+
|window                                    |city  |count|
+------------------------------------------+------+-----+
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Tanger|1    |
+------------------------------------------+------+-----+

-------------------------------------------
Batch: 20
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-00049

-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |city      |count|
+------------------------------------------+----------+-----+
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Marrakech |26   |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Rabat     |3    |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Tanger    |25   |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Rabat     |1    |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Rabat     |36   |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Casablanca|62   |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Marrakech |3    |
+------------------------------------------+----------+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+----------+---------+-----+
|city      |status   |count|
+----------+---------+-----+
|Casablanca|CANCELLED|10   |
|Casablanca|CREATED  |117  |
|Casablanca|DELIVERED|1    |
|Casablanca|PICKED_UP|194  |
|Marrakech |CANCELLED|7    |
|Marrakech |CREATED  |50   |
|Marrakech |PICKED_UP|92   |
|Rabat     |CANCELLED|7    |
|Rabat     |CREATED  |69   |
|Rabat     |PICKED_UP|95   |
|Tanger    |CANCELLED|7    |
|Tanger    |CREATED  |36   |
|Tanger    |DELIVERED|1    |
|Tanger    |PICKED_UP|81   |
+----------+---------+-----+

-------------------------------------------
Batch: 21
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_ti

In [7]:
import json, time
from datetime import datetime, timedelta, timezone
from kafka import KafkaProducer

def etat_watermark():
    p = q_wm.lastProgress
    return (p["batchId"], p["eventTime"].get("watermark"),
            p["stateOperators"][0]["numRowsDroppedByWatermark"])

print("Avant  : batch, watermark, lignes rejetées =", etat_watermark())

late_time = datetime.now(timezone.utc) - timedelta(minutes=15)
evt = {"order_id": "ORD-LATE-0001", "restaurant_id": "R-001", "customer_id": "C-00001",
       "city": "Casablanca", "status": "CREATED", "amount_mad": 100.0, "items_count": 1,
       "distance_km": 2.0, "courier_id": "L-CAS-01",
       "event_time": late_time.strftime("%Y-%m-%dT%H:%M:%SZ")}
prod = KafkaProducer(bootstrap_servers="kafka:9092",
                     value_serializer=lambda v: json.dumps(v).encode())
prod.send("orders", evt); prod.flush()
print("Envoyé : event_time =", evt["event_time"], "(15 min de retard)")

time.sleep(35)
print("Après  : batch, watermark, lignes rejetées =", etat_watermark())

+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0003877|R-112        |C-05767    |Casablanca|PICKED_UP|107.1     |2          |1.9        |L-CAS-08  |2026-11-15 18:33:40|2        |4199  |2026-11-15 18:33:40.267|
|ORD-0008966|R-034        |C-01869    |Casablanca|CREATED  |109.73    |2          |1.4        |L-CAS-03  |2026-11-15 18:33:40|2        |4200  |2026-11-15 18:33:40.524|
|ORD-0003979|R-114        |C-02474    |Rabat     |PICKED_UP|206.24    |4          |2.1        |L-RAB-06  |2026-11-15 18:33:40|2        |4201  |2026-11-15 18:33:

-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |city      |count|
+------------------------------------------+----------+-----+
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Marrakech |151  |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Casablanca|3    |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Tanger    |110  |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Rabat     |7    |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Rabat     |3    |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Rabat     |195  |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Casablanca|305  |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Marrakech |7    |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Marrakech |1    |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Casablanca|3    |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Tanger    |3    |
+----------------------------------

-------------------------------------------
Batch: 7
-------------------------------------------
+----------+---------+-----+
|city      |status   |count|
+----------+---------+-----+
|Casablanca|CANCELLED|35   |
|Casablanca|CREATED  |305  |
|Casablanca|DELIVERED|5    |
|Casablanca|PICKED_UP|466  |
|Marrakech |CANCELLED|20   |
|Marrakech |CREATED  |141  |
|Marrakech |DELIVERED|6    |
|Marrakech |PICKED_UP|251  |
|Rabat     |CANCELLED|27   |
|Rabat     |CREATED  |182  |
|Rabat     |DELIVERED|6    |
|Rabat     |PICKED_UP|269  |
|Tanger    |CANCELLED|16   |
|Tanger    |CREATED  |103  |
|Tanger    |DELIVERED|4    |
|Tanger    |PICKED_UP|192  |
+----------+---------+-----+



In [8]:
for q in spark.streams.active:
    p = q.lastProgress or {}
    print(f"{q.name:<16} actif={q.isActive}  batch={p.get('batchId')}  "
          f"lignes={p.get('numInputRows')}  débit={p.get('processedRowsPerSecond')}")

p5_counts        actif=True  batch=7  lignes=298  débit=181.37553256238587
p5_console_brut  actif=True  batch=26  lignes=196  débit=196000.0
p5_watermark     actif=True  batch=5  lignes=299  débit=386.8046571798189
p5_bronze        actif=True  batch=5  lignes=628  débit=628000.0
-------------------------------------------
Batch: 27
-------------------------------------------
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|order_id   |restaurant_id|customer_id|city      |status   |amount_mad|items_count|distance_km|courier_id|event_time         |partition|offset|kafka_ts               |
+-----------+-------------+-----------+----------+---------+----------+-----------+-----------+----------+-------------------+---------+------+-----------------------+
|ORD-0001522|R-188        |C-02515    |Marrakech |PICKED_UP|73.62     |1          |1.2        |L-MAR-01  |2026-11-15 1

-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------+-----+
|window                                    |city      |count|
+------------------------------------------+----------+-----+
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Marrakech |360  |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Tanger    |252  |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Rabat     |4    |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Rabat     |422  |
|{2026-11-15 18:30:00, 2026-11-15 18:35:00}|Casablanca|652  |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Marrakech |14   |
|{2026-11-15 18:25:00, 2026-11-15 18:30:00}|Casablanca|11   |
|{2026-11-15 18:20:00, 2026-11-15 18:25:00}|Tanger    |1    |
+------------------------------------------+----------+-----+

-------------------------------------------
Batch: 8
-------------------------------------------
+----------+---------+-----+
|city      |status   |count|
+--

In [9]:
for q in spark.streams.active:
    q.stop()
spark.stop()

In [10]:
print(f"{'batch':>5} | {'watermark':<26} | {'lignes lues':>11} | {'rejetées (watermark)':>20}")
for p in q_wm.recentProgress:
    print(f"{p['batchId']:>5} | {str(p['eventTime'].get('watermark')):<26} | "
          f"{p['numInputRows']:>11} | {p['stateOperators'][0]['numRowsDroppedByWatermark']:>20}")

batch | watermark                  | lignes lues | rejetées (watermark)
    0 | 1970-01-01T00:00:00.000Z   |           1 |                    0
    1 | 2026-11-15T18:23:22.000Z   |         155 |                    0
    2 | 2026-11-15T18:23:29.000Z   |         325 |                    0
    3 | 2026-11-15T18:23:44.000Z   |         308 |                    1
    4 | 2026-11-15T18:23:59.000Z   |         330 |                    0
    5 | 2026-11-15T18:24:14.000Z   |         299 |                    0
    6 | 2026-11-15T18:24:29.000Z   |         325 |                    0


## Partie 5 – Analyse

### Modèle source → transformation → sink (table infinie)
Structured Streaming traite un flux comme une **table qui grandit sans fin** : chaque message Kafka devient une nouvelle ligne ajoutée à la table d'entrée. On écrit la requête comme sur un DataFrame batch (`from_json`, `groupBy`, `window`), et Spark l'exécute de manière **incrémentale**.
- **Source** : `readStream.format("kafka")` sur le topic `orders`. Kafka fournit `value` en binaire, ainsi que `partition`, `offset` et `timestamp`.
- **Transformation** : `from_json` avec le schéma de la Partie 3, puis agrégations.
- **Sink** : `console` pour l'affichage, et fichiers JSON dans `lake/bronze/orders_raw/` avec un checkpoint dédié.

### Micro-batch ou traitement continu
- **Micro-batch** (`trigger(processingTime="10 seconds")`, notre choix) : toutes les 10 s, Spark lit les nouveaux offsets, exécute un petit job batch et écrit le résultat. La latence est de l'ordre de la seconde. Toutes les opérations sont supportées (agrégations, jointures, watermark) et la garantie exactly-once est possible. On voit les batches 0, 1, 2… dans la console. Le message « Current batch is falling behind » signale qu'un batch a duré plus longtemps que l'intervalle (10,3 s).
- **Continu** (`trigger(continuous="1 second")`) : des tâches longues lisent les messages en continu, pour une latence d'environ 1 ms. Seules les opérations sans état sont permises (map, filter, pas d'agrégation), avec une garantie at-least-once.

### Les trois modes de sortie
| Mode | Ce qui est écrit à chaque batch | Notre usage |
|---|---|---|
| `append` | uniquement les **nouvelles lignes** définitives | flux brut (console, Bronze) |
| `complete` | **toute** la table de résultat | comptage par ville et statut : la table entière est réécrite, avec des compteurs croissants |
| `update` | uniquement les lignes **modifiées** depuis le dernier batch | fenêtres de 5 min avec watermark |

`append` sur une agrégation n'est possible qu'avec un watermark : la ligne n'est écrite qu'une fois sa fenêtre fermée.

### Watermark de 10 minutes et événement en retard de 15 minutes
`withWatermark("event_time", "10 minutes")` définit **watermark = event_time maximum vu − 10 min**. Spark garde en mémoire l'état des fenêtres plus récentes que le watermark et **rejette** les événements plus anciens.

Expérience : un événement `ORD-LATE-0001` est envoyé avec `event_time = 18:18:53`, soit 15 min de retard, alors que le watermark vaut **18:23:29**. Au batch suivant (batch 3), `numRowsDroppedByWatermark = 1` : l'événement est **ignoré** et n'est compté dans aucune fenêtre. Un événement en retard de moins de 10 min (par exemple ORD-0001726, `event_time 18:24:52` reçu à 18:31:31) est au contraire accepté et met à jour sa fenêtre `[18:20, 18:25]` en mode update.

Le watermark est un **compromis** : plus il est long, plus on tolère de retard, mais plus l'état à garder en mémoire est gros et plus les résultats définitifs arrivent tard.